## Interval Analysis

In [10]:
# !pip install tensorboardX
# !pip install bound-propagation

from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
bound_test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

class Net(nn.Sequential):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(28*28, 50)
        self.fc2 = nn.Linear(50, 50)
        self.fc3 = nn.Linear(50, 50)
        self.out = nn.Linear(50, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.out(x)
        return x


stan_model = Net().to(device)
stan_model.train()


Net(
  (fc1): Linear(in_features=784, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (out): Linear(in_features=50, out_features=10, bias=True)
)

In [11]:
def train_model(model, num_epochs=15, lr=1e-3):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for batch_idx, (images, labels) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(images), labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.3f}')

def test_model(model):
    model.eval()
    
    with torch.no_grad():
        correct = 0
        total = 0
        for data in test_loader:
            images, labels = data
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f'Accuracy on images: {100 * correct / total}')
startq = time.time()
train_model(stan_model)
endq = time.time()

print(f"Standard Training Time: {endq - startq} seconds")
print("Standard Model Test Performance:")
test_model(stan_model)

Epoch 1/15: 100%|██████████| 938/938 [00:02<00:00, 447.04it/s]


Epoch 1/15, Loss: 0.427


Epoch 2/15: 100%|██████████| 938/938 [00:02<00:00, 463.38it/s]


Epoch 2/15, Loss: 0.182


Epoch 3/15: 100%|██████████| 938/938 [00:02<00:00, 460.36it/s]


...

Epoch 13/15, Loss: 0.039


Epoch 14/15: 100%|██████████| 938/938 [00:02<00:00, 454.04it/s]


Epoch 14/15, Loss: 0.034


Epoch 15/15: 100%|██████████| 938/938 [00:02<00:00, 454.83it/s]


Epoch 15/15, Loss: 0.033
Standard Training Time: 30.787933588027954 seconds
Standard Model Test Performance:
Accuracy on images: 97.2


In [12]:
from bound_propagation import BoundModelFactory, HyperRectangle
def rb_worst(model, x, y, eps):
    x = x.view(-1, 28*28)
    L = torch.clamp(x - eps, 0, 1)
    U = torch.clamp(x + eps, 0, 1)
    factory = BoundModelFactory()
    box = HyperRectangle(L, U)

    alter = nn.Sequential(
        model.fc1,
        nn.ReLU(),
        model.fc2,
        nn.ReLU(),
        model.fc3,
        nn.ReLU(),
        model.out   
    )

    bound_net = factory.build(alter)
    interval = bound_net.ibp(box)
    lower, higher = interval.lower, interval.upper
    mask = torch.ones_like(lower, dtype=torch.bool)
    for i in range(x.size(0)):
        mask[i, y[i]] = False
    res = torch.where(mask, higher, lower)
    loss = F.cross_entropy(res, y)
    return loss

In [13]:
def train_model_ibp(model, num_epochs):
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(num_epochs):
        model.train()
        kappa = 1.0 - 0.5 * (epoch / num_epochs)
        eps = 0.1 * epoch / num_epochs
        for batch_idx, (images, labels) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs}"):
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            logits = model(images)
            ce_loss = F.cross_entropy(logits, labels)
            rb_loss = rb_worst(model, images, labels, eps)
            loss = kappa * ce_loss + (1 - kappa) * rb_loss
            loss.backward()
            optimizer.step()
        model.eval()
        tot_val, tot_acc = 0.0, 0.0
        val_loss = 0.0
        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            with torch.no_grad():
                logits = model(images)
                ce_loss = F.cross_entropy(logits, labels)
                rb_loss = rb_worst(model, images, labels, eps)
                loss = kappa * ce_loss + (1 - kappa) * rb_loss
                val_loss += loss.item()
                tot_acc += (logits.argmax(dim=1) == labels).sum().item()
                tot_val += labels.size(0)
        val_acc = 100.0 * tot_acc / tot_val

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {val_loss/len(train_loader):.3f}, Accuracy: {val_acc:.2f}%')
ibp_model = Net().to(device)
ibp_model.train()
start = time.time()
train_model_ibp(ibp_model, 15)
end = time.time()

print(f"IBP Training Time: {end - start} seconds")

Epoch 1/15: 100%|██████████| 938/938 [00:04<00:00, 200.86it/s]


Epoch 1/15, Loss: 0.215, Accuracy: 93.64%


Epoch 2/15: 100%|██████████| 938/938 [00:04<00:00, 199.25it/s]


Epoch 2/15, Loss: 0.197, Accuracy: 95.10%


Epoch 3/15: 100%|██████████| 938/938 [00:04<00:00, 201.50it/s]


...

Epoch 13/15, Loss: 0.413, Accuracy: 95.62%


Epoch 14/15: 100%|██████████| 938/938 [00:04<00:00, 208.14it/s]


Epoch 14/15, Loss: 0.464, Accuracy: 95.36%


Epoch 15/15: 100%|██████████| 938/938 [00:04<00:00, 199.00it/s]


Epoch 15/15, Loss: 0.512, Accuracy: 94.94%
IBP Training Time: 117.28096413612366 seconds


In [9]:
def pgd(model, x, labels, k=40, eps=8/255, eps_step=0.001):
    model.eval()
    ce_loss = F.cross_entropy
    adv_x = x.clone().detach()
    adv_x = torch.clamp(adv_x, 0.0, 1.0)
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        logits = model(adv_x)
        # TODO: Calculate the loss
        loss = ce_loss(logits, labels)
        loss.backward()
        assert adv_x.grad is not None
        grad = adv_x.grad.data
        # TODO: compute the adv_x
        adv_x = adv_x.detach() + eps_step * torch.sign(grad)
        adv_x = torch.clamp(adv_x, x - eps, x + eps)
        adv_x = torch.clamp(adv_x, 0, 1).detach()
    return adv_x

def evaluate_pgd(model):
    model.eval()
    tot_acc = 0.0
    tot_test = 0.0
    tot_acc_adv = 0.0
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)
        images = images.view(-1, 28*28)
        adv_images = pgd(model, images, labels)
        outputs = model(adv_images)
        tot_acc_adv += (outputs.argmax(dim=1) == labels).sum().item()
        tot_test += labels.size(0)
        tot_acc += (model(images).argmax(dim=1) == labels).sum().item()
    return 100.0 * tot_acc / tot_test, 100.0 * tot_acc_adv / tot_test

stan_acc, stan_acc_adv = evaluate_pgd(stan_model)
print(f'Standard Model - Clean Accuracy: {stan_acc:.2f}%, PGD Accuracy: {stan_acc_adv:.2f}%')
ibp_acc, ibp_acc_adv = evaluate_pgd(ibp_model)
print(f'IBP Model - Clean Accuracy: {ibp_acc:.2f}%, PGD Accuracy: {ibp_acc_adv:.2f}%')


Standard Model - Clean Accuracy: 97.26%, PGD Accuracy: 57.33%
IBP Model - Clean Accuracy: 95.89%, PGD Accuracy: 91.54%


### Write the interval analysis for the simple model

In [ ]:
## TODO: Write the interval analysis for the simple model
## you can use https://github.com/Zinoex/bound_propagation

from bound_propagation import BoundModelFactory, HyperRectangle

factory = BoundModelFactory()
isinstance(stan_model, nn.Sequential)
net = factory.build(stan_model)

In [ ]:
net.eval()
epsilons = np.linspace(0.01, 0.1, 10)
always_correct = np.zeros(len(epsilons), dtype=int)
still_correct = np.zeros(len(epsilons), dtype=int)
correct = 0
total = 0

for images, labels in test_loader:
    images = images.view((-1, 28 * 28)).to(device)
    labels = labels.to(device)

    outputs = stan_model(images)
    _, predicted = torch.max(outputs.data, 1)
    predicted_correct = predicted == labels

    total += labels.size(0)
    correct += predicted_correct.sum().item()

    for idx, epsilon in enumerate(epsilons):
        input_bounds = HyperRectangle.from_eps(images, float(epsilon))
        crown_ibp_bounds = net.crown_ibp(input_bounds).concretize()
        lower, upper = crown_ibp_bounds.lower, crown_ibp_bounds.upper

        label_lower = lower.gather(1, labels.unsqueeze(1)).squeeze(1)
        label_mask = torch.zeros_like(upper, dtype=torch.bool)
        label_mask.scatter_(1, labels.unsqueeze(1), True)
        max_other_upper = upper.masked_fill(label_mask, float('-inf')).max(dim=1).values
        robust_mask = label_lower > max_other_upper
        always_correct[idx] += robust_mask.sum().item()

print(f'Standard accuracy: {correct / total*100:.2f}%')
for eps, ac in zip(epsilons, always_correct):
    print(f'eps={eps:.3f}: always_correct={ac / total * 100:.2f}%')

        
        
        
        
    


Standard accuracy: 96.70%
eps=0.010: always_correct=42.67%
eps=0.020: always_correct=18.58%
eps=0.030: always_correct=5.53%
eps=0.040: always_correct=1.23%
eps=0.050: always_correct=0.27%
eps=0.060: always_correct=0.08%
eps=0.070: always_correct=0.03%
eps=0.080: always_correct=0.00%
eps=0.090: always_correct=0.00%
eps=0.100: always_correct=0.00%
